In [11]:
from typing import TypedDict
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import BaseMessage

In [12]:
from typing import List, Dict, Any, Literal
from typing_extensions import Annotated, TypedDict
from langgraph.graph import MessagesState
from langgraph.types import Command
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import SystemMessage, HumanMessage , AIMessage

from backend.chatbot.prompts.router.supervisor_prompt import prompt as supervisor_prompt

def merge_dicts(old, new): # i used this to let collected output gest updated before overwriting
    return {**old, **new}

class GraphState(MessagesState):
    remaining_agents: List[str]
    collected_outputs: Annotated[Dict[str, Any], merge_dicts]
    final_answer: str


class RouterState(TypedDict):
    next: str
    reason: str
    confidence: float

agents_description = """
        attachment: Detects attachment style patterns (secure/anxious/avoidant/disorganized) from relational signals.
        clinical_disorder: Detects possible clinical syndrome patterns from symptom clusters.
        cognetive_distortion: Detects distorted thinking patterns (catastrophizing, black-and-white thinking, etc.).
        functional_level: Assesses impairment in work, social, self-care, and daily functioning.
        personal_traits: Detects stable personality trait tendencies from repeated behavioral-emotional patterns.
        relational_pattern: Detects interpersonal dynamics, boundaries, dependency, conflict, and communication patterns.
        schema: Detects early maladaptive schemas and core beliefs.
""".strip()


In [13]:
import os
from langchain_openai import ChatOpenAI
from user_context import first, second, third


# llm = ChatOpenAI(
#     model="deepseek-v4-flash",
#     api_key='sk-1666caf6a268456d8df5b5da9853d5d6',
#     base_url="https://api.deepseek.com",
#     temperature=0,
#     reasoning_effort="high",",

# llm = ChatOpenAI(
#     model="deepseek-v4-flash",
#     api_key="sk-1666caf6a268456d8df5b5da9853d5d6",
#     base_url="https://api.deepseek.com",
#     temperature=0,
#     reasoning_effort="high",
#     extra_body={
#         "thinking": {"type": "enabled"},
#         # "response_format": {"type": "json_object"} 
#     }
# )

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key="sk-1666caf6a268456d8df5b5da9853d5d6",
    base_url="https://api.deepseek.com",
    temperature=0,
    reasoning_effort="high",

    model_kwargs={
        "response_format": {
            "type": "json_object"
        }
    },

    extra_body={
        "thinking": {
            "type": "enabled"
        }
    }
)

# from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(
#     base_url="https://api.gapgpt.app/v1",
#     api_key="sk-maGdVnAynciq7MyrhlnX6NrVYcPirPgNR1y8N5CcxglcEVWG",
#     model="gpt-5.4",
#     temperature=0
# )

### agent list

In [14]:

from backend.chatbot.prompts.attachment import prompt as attachment_prompt
from backend.chatbot.prompts.schema import prompt as  schema_prompt 
from backend.chatbot.prompts.clinical_disorder import prompt as clinical_disorder_prompt 
from backend.chatbot.prompts.personal_traits import prompt as personal_train_prompt 
from backend.chatbot.prompts.relational_pattern import prompt as relational_pattern_prompt
from backend.chatbot.prompts.functional_level import prompt as functional_level_prompt
from backend.chatbot.prompts.cognetive_distortion import prompt as cognitive_distortation_prompt
import json

def create_agent_node(llm  , prompt , node_name) :
    
    def agent(state:GraphState)-> Command[Literal['supervisor']]: 
        user_context = state["messages"][-1].content + " "
        messages = [
            SystemMessage(content= prompt), 
            HumanMessage(content = user_context)
        ]

        response = llm.invoke(messages) # router node 
        print(response.response_metadata['token_usage'])

        # updated_messages = state['messages'] + [
        #     AIMessage(content = response.content , name = node_name)
        # ]
        try: 
            data = json.loads(response.content)
            print('-'*50)
            print(f'agent {node_name} called , response is :',data)
            print('-'*50)

        except: 
            print('json error raw output is ' , response.content)

        return Command(
            update={
                "collected_outputs": {
                    node_name: data
                }
            },
            goto="supervisor"
        )
    return agent

attachment_agent = create_agent_node(llm, attachment_prompt , node_name = 'atthchment')
schema_agent = create_agent_node(llm, schema_prompt , node_name = 'schema')
clinical_disorder_agent = create_agent_node(llm, clinical_disorder_prompt , node_name = 'clinical_disorder')
personal_trait_agent = create_agent_node(llm, personal_train_prompt , node_name = 'personal_trait')
relational_pattern_agent = create_agent_node(llm, relational_pattern_prompt , node_name = 'relational_pattern')
functional_level_agent = create_agent_node(llm, functional_level_prompt , node_name = 'functional_level')
cognitive_distortion_agent = create_agent_node(llm, cognitive_distortation_prompt , node_name = 'cognitive_distortion')

# from tenacity import retry, stop_after_attempt, wait_exponential

# @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
# def safe_llm_invoke(prompt):
#     return llm.invoke(prompt)

In [15]:
llm.invoke('  json hi سلام')

AIMessage(content='{"response": "hi سلام"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 28, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 109, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 28}, 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'f9be77f1-b4cc-4982-92b8-8b30e8509683', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7fc0-42e3-7b91-9840-9c4c9654792c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 117, 'total_tokens': 145, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 109}})

(با ورودی ~1850 و خروجی ~400)


### planner and executor code 

In [ ]:
from typing import List, Dict, Any
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, END
from langgraph.types import Command
from langgraph.graph import MessagesState
from backend.chatbot.prompts.meta_analist import prompt as meta_analist_prompt
from user_context import first, second, third
import json

def router_node(state: GraphState) -> Command[
       Literal['supervisor']
    ]:
        remaining_agents = state.get("remaining_agents", [])
        if not remaining_agents:
            return Command(goto="meta_analist")

        options = remaining_agents 

        system_prompt = (
              supervisor_prompt
              + "\n\nAvailable agents: " + ", ".join(options)
              + "\n\nAgent descriptions:\n" + agents_description
            )
        user_context = state["messages"][-1].content

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_context),
        ]

        response = llm.invoke(messages)
        # print(response.response_metadata['token_usage'])

        data = json.loads(response.content)
        print('*'*50)
        print("router node call" , response.response_metadata['token_usage'])
        print('*'*50)
        return Command(
            goto = 'supervisor',
            update = {
                'remaining_agents' : data['selected_agents'],
                'collected_outputs': {} 
            }
        )


def supervisor_node(state: GraphState) -> Command[
    Literal[
        "attachment",
        "schema",
        "clinical_disorder",
        "personal_traits",
        "functional_level",
        "cognetive_distortion",
        "relational_pattern",
        "meta_analist", ]
        ]:
    remaining_agents = state.get("remaining_agents", [])

    if not remaining_agents:
        return Command(goto="meta_analist")

    next_agent = remaining_agents[0]
    # print('/'*50)
    # print(f'supervisor node is called , next_agent is' , next_agent)
    # print('/'*50)
    return Command(
        goto=next_agent,
        update={
            "remaining_agents": remaining_agents[1:],
            # 'remaining_agents' : []  
        },
    )

def meta_analist_node(state: GraphState) -> Command:
    outputs = state.get("collected_outputs", {})
    client_message = state["messages"][-1].content
    payload = {
        "client_message": client_message,
        "agent_outputs": outputs,
    }
    messages = [
        SystemMessage(content=meta_analist_prompt),
        HumanMessage(content=json.dumps(payload, ensure_ascii=False)),
    ]
    response = llm.invoke(messages)
    try: 
        result = json.loads(response.content)
        print('*'*50)
        print('meta analist' , response.response_metadata['token_usage'])
        print('*'*50)
      
    except: 
        print('error happening , raw data ', response.content)

    return Command(
        goto=END,
        update={"final_answer": result},
    )


builder = StateGraph(GraphState)

builder.add_node("router", router_node)
builder.add_node("supervisor", supervisor_node)

agent_info = {
    "attachment" : attachment_prompt,
    "schema" : schema_prompt,
    "clinical_disorder" : clinical_disorder_prompt,
    "personal_traits" : personal_train_prompt,
    "functional_level" : functional_level_prompt,
    'cognetive_distortion' : cognitive_distortation_prompt,
    'relational_pattern' : relational_pattern_prompt
}

for name , prompt in agent_info.items():
    builder.add_node(name, create_agent_node(llm, prompt , node_name = name))

builder.add_node("meta_analist", meta_analist_node)

builder.set_entry_point("router")

graph = builder.compile()

# result = graph.invoke(
#     {
#         "messages": [],
#     }
# )

from IPython.display import Image, display

# img = graph.get_graph().draw_mermaid_png()
# display(Image(img))
# print(graph.get_graph().draw_mermaid())

In [17]:
from backend.chatbot.prompts import emotional_state, personal_traits
from user_context import forth

state = {
    "messages": [
        HumanMessage(content=forth) 
    ],
    "remaining_agents": [
        "schema",
        "attachment",
        "clinical_disorder",
        "cognetive_distortion",
        "functional_level",
        "personal_traits",
        "relational_pattern",
    ],
    "collected_outputs": {},
    "final_answer": None,  
}

result  = graph.invoke(state)
result

# result = schema_agent(state)

**************************************************
router node call {'completion_tokens': 742, 'prompt_tokens': 1118, 'total_tokens': 1860, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 535, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1024}, 'prompt_cache_hit_tokens': 1024, 'prompt_cache_miss_tokens': 94}
**************************************************
{'completion_tokens': 1349, 'prompt_tokens': 1209, 'total_tokens': 2558, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 774, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1152}, 'prompt_cache_hit_tokens': 1152, 'prompt_cache_miss_tokens': 57}
--------------------------------------------------
agent attachment called , response is : {'Secure': {'score': 0, 'evidence': [], 'clinical_analysis': 'هیچ نشانه

{'messages': [HumanMessage(content='\nمن رابطه ی عاطفی رو به خیانت کردن خودم از دست داام ،  و یه حسرت بزرگی از اون رابطه تو دلم هست ، با اینکه طرف ویژگی های بد بسیار زیادی داشت ، از جمله نقص و شرم خیلی زیاد داشت ، \nو با اینکه آگاهم نسبت به عیباش و اون حسرت تو دلم مونده ،با اینکه وارد یه رابط ی جدیدی شدم ، که تقریبا مطمئنم این رابطه موقته ، و می دونم که یروز نزدیکی مسیرش جدا میشه \nاما بازم دارم جوی زندگی می کنم که به خودم آسیب می زنم ، اینجوریه که دارم ازون تایید میگیرم ، تمام پولم رو براش تقریبا خرج می کنم ، وقتی که می توونم صرف ساز زدن و پیشرفت بکنم دارم با اون می گذرونم \nدوست دارم یجوری لباس بپوشم که اون خوشش بیاد ، نسبت به اینکه بدن وزشکاری ندارم حس مضخرفی دارم ، چون اوون ورزشگار خوشش میاد ، نسبت به اینکه درامد زیادی ندارم حس مضخرفی دارم\nچون اون خوشش نمیاد که من پول تداشته باشم ، \nدائم در حال تایید گرفتن هستم ازش ، بعضی وقتا وقتی دائم دارم این پولو براش خرج می کنم به خودم فوش می دم\nکه چرا این پول رو دادی ، به خودم میگم این پول رو می تونستی برا خودت کفش بخری ، در صورتی که کفش ن

In [18]:
result

{'messages': [HumanMessage(content='\nمن رابطه ی عاطفی رو به خیانت کردن خودم از دست داام ،  و یه حسرت بزرگی از اون رابطه تو دلم هست ، با اینکه طرف ویژگی های بد بسیار زیادی داشت ، از جمله نقص و شرم خیلی زیاد داشت ، \nو با اینکه آگاهم نسبت به عیباش و اون حسرت تو دلم مونده ،با اینکه وارد یه رابط ی جدیدی شدم ، که تقریبا مطمئنم این رابطه موقته ، و می دونم که یروز نزدیکی مسیرش جدا میشه \nاما بازم دارم جوی زندگی می کنم که به خودم آسیب می زنم ، اینجوریه که دارم ازون تایید میگیرم ، تمام پولم رو براش تقریبا خرج می کنم ، وقتی که می توونم صرف ساز زدن و پیشرفت بکنم دارم با اون می گذرونم \nدوست دارم یجوری لباس بپوشم که اون خوشش بیاد ، نسبت به اینکه بدن وزشکاری ندارم حس مضخرفی دارم ، چون اوون ورزشگار خوشش میاد ، نسبت به اینکه درامد زیادی ندارم حس مضخرفی دارم\nچون اون خوشش نمیاد که من پول تداشته باشم ، \nدائم در حال تایید گرفتن هستم ازش ، بعضی وقتا وقتی دائم دارم این پولو براش خرج می کنم به خودم فوش می دم\nکه چرا این پول رو دادی ، به خودم میگم این پول رو می تونستی برا خودت کفش بخری ، در صورتی که کفش ن

In [19]:
result.update['collected_outputs']['schema']['scores']


TypeError: 'builtin_function_or_method' object is not subscriptable

In [ ]:
summary = result.update['collected_outputs']['schema']['summary']
print(summary)

In [ ]:
result.update['collected_outputs']['schema']['evidence']

In [ ]:
result['collected_outputs'].keys()

### مراجع تست ۲

خروجی `graph.invoke` روی `user_context.forth` — ذخیره‌شده در `content_output/context_output2.json` و wired در frontend به `test-patient-2` / «مراجع تست ۲».

In [ ]:
import json
from pathlib import Path

REFERENCES_TEST_2 = json.loads(
    Path("content_output/context_output2.json").read_text(encoding="utf-8")
)

# collected_outputs از همه agentها + final_answer از meta_analist
REFERENCES_TEST_2["collected_outputs"].keys()
REFERENCES_TEST_2["final_answer"]["json_scores"]

options we have 

first -> summorize and run agents 
second -> use one agent for all agents no route and graph 
